# Voxen CAD Model Fine-Tuning with Unsloth

This notebook trains a `Qwen2.5-Coder-1.5B` model to output structurally valid JSON for our Next.js CAD Viewer based on our strict `AssemblySchema`.

**Hardware Requirements:**
- 1x GPU (T4, L4, A100, or MI300X)
- ~12GB system RAM

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

### 1. Load Model & Tokenizer
We use the `1.5B Instruct` model for maximum local CPU inference speed.

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Supports up to 4096, but 2048 uses less VRAM
dtype = None # Auto detection. Float16 for Tesla T4, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-Coder-1.5B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

### 2. Apply LoRA Adapters
We only update 1-10% of all parameters, which makes training extremely fast.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

### 3. Load Dataset
Make sure you upload the `dataset.jsonl` generated by `dataset_generator.py` to the notebook environment.

In [ ]:
from datasets import load_dataset

# Apply the Chat Template automatically
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
    mapping = {"role" : "role", "content" : "content", "user" : "user", "assistant" : "assistant"}
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }

dataset = load_dataset("json", data_files="dataset.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched = True,)

print(dataset[0]["text"])

### 4. Train the Model

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Increase this to 300-500 for a real training run!
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
    ),
)

trainer_stats = trainer.train()

### 5. Export to GGUF
Exporting directly to a 4-bit quantized GGUF so you can run it locally in your Python backend!

In [ ]:
# Save to 4bit Q4_K_M GGUF
model.save_pretrained_gguf("finetuned_model", tokenizer, quantization_method = "q4_k_m")

print("✅ Finished! Download `finetuned_model-unsloth-Q4_K_M.gguf` from the file explorer on the left and place it in your local models folder.")